# DSC540 Final Project Milestone 2 
## Komal Shahid

### Exploring Women in Tech: A Data Wrangling Study 

In [1]:
#imported libraries 
import pandas as pd
import numpy as np
import re
from fuzzywuzzy import fuzz
from scipy import stats
from fuzzywuzzy import process

/Users/komalshahid/opt/anaconda3/lib/python3.8/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [2]:
# Read the Kaggle Survey dataset into a DataFrame
df = pd.read_csv('kaggle-survey-2021/kaggle_survey_2021_responses.csv', low_memory= False)
df.head()

,Time from Start to Finish (seconds),Q1,Q2,Q3,Q4,Q5,Q6,Q7_Part_1,Q7_Part_2,Q7_Part_3,...,Q38_B_Part_3,Q38_B_Part_4,Q38_B_Part_5,Q38_B_Part_6,Q38_B_Part_7,Q38_B_Part_8,Q38_B_Part_9,Q38_B_Part_10,Q38_B_Part_11,Q38_B_OTHER
0,Duration (in seconds),What is your age (# years)?,What is your gender? - Selected Choice,In which country do you currently reside?,What is the highest level of formal education ...,Select the title most similar to your current ...,For how many years have you been writing code ...,What programming languages do you use on a reg...,What programming languages do you use on a reg...,What programming languages do you use on a reg...,...,"In the next 2 years, do you hope to become mor...","In the next 2 years, do you hope to become mor...","In the next 2 years, do you hope to become mor...","In the next 2 years, do you hope to become mor...","In the next 2 years, do you hope to become mor...","In the next 2 years, do you hope to become mor...","In the next 2 years, do you hope to become mor...","In the next 2 years, do you hope to become mor...","In the next 2 years, do you hope to become mor...","In the next 2 years, do you hope to become mor..."
1,910,50-54,Man,India,Bachelor’s degree,Other,5-10 years,Python,R,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,784,50-54,Man,Indonesia,Master’s degree,Program/Project Manager,20+ years,NaN,NaN,SQL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN
3,924,22-24,Man,Pakistan,Master’s degree,Software Engineer,1-3 years,Python,NaN,NaN,...,NaN,NaN,TensorBoard,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,575,45-49,Man,Mexico,Doctoral degree,Research Scientist,20+ years,Python,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN


#### STEP 1: Drop the first row from each column because the data starts from row 1 .

In [3]:
## Since row 0 is the question being asked in the survey
df = df.iloc[1:]
df.head()

,Time from Start to Finish (seconds),Q1,Q2,Q3,Q4,Q5,Q6,Q7_Part_1,Q7_Part_2,Q7_Part_3,...,Q38_B_Part_3,Q38_B_Part_4,Q38_B_Part_5,Q38_B_Part_6,Q38_B_Part_7,Q38_B_Part_8,Q38_B_Part_9,Q38_B_Part_10,Q38_B_Part_11,Q38_B_OTHER
1,910,50-54,Man,India,Bachelor’s degree,Other,5-10 years,Python,R,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,784,50-54,Man,Indonesia,Master’s degree,Program/Project Manager,20+ years,NaN,NaN,SQL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN
3,924,22-24,Man,Pakistan,Master’s degree,Software Engineer,1-3 years,Python,NaN,NaN,...,NaN,NaN,TensorBoard,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,575,45-49,Man,Mexico,Doctoral degree,Research Scientist,20+ years,Python,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN
5,781,45-49,Man,India,Doctoral degree,Other,< 1 years,Python,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### step2:  Shorten the dataset to keep the columns of interest only

In [4]:
columns_to_keep = ['Q1', 'Q2', 'Q3','Q4','Q5','Q6',
                   'Q7_Part_1','Q7_Part_2','Q7_Part_3','Q7_Part_4','Q7_Part_5',
                   'Q7_Part_6','Q7_Part_7','Q7_Part_8','Q7_Part_9','Q7_Part_10','Q7_Part_11',
                   'Q7_Part_12','Q7_OTHER',
                   'Q20','Q21','Q25']  
subset_df = df[columns_to_keep]
# Print the subset DataFrame
print(subset_df.head())

      Q1   Q2         Q3                 Q4                       Q5  \
1  50-54  Man      India  Bachelor’s degree                    Other   
2  50-54  Man  Indonesia    Master’s degree  Program/Project Manager   
3  22-24  Man   Pakistan    Master’s degree        Software Engineer   
4  45-49  Man     Mexico    Doctoral degree       Research Scientist   
5  45-49  Man      India    Doctoral degree                    Other   

           Q6 Q7_Part_1 Q7_Part_2 Q7_Part_3 Q7_Part_4  ... Q7_Part_7  \
1  5-10 years    Python         R       NaN       NaN  ...       NaN   
2   20+ years       NaN       NaN       SQL         C  ...       NaN   
3   1-3 years    Python       NaN       NaN       NaN  ...       NaN   
4   20+ years    Python       NaN       NaN       NaN  ...       NaN   
5   < 1 years    Python       NaN       NaN         C  ...       NaN   

  Q7_Part_8 Q7_Part_9 Q7_Part_10 Q7_Part_11 Q7_Part_12 Q7_OTHER  \
1       NaN       NaN        NaN        NaN        NaN      NaN   


####  step 3: Combine the column whose answer is split into multiple columns 

In [15]:
# Create a new column by combining multiple columns based on another column
## form the dataset, you can see that question 7 has multiple parts and even though it is string data type because of nan values,
## there is inconsistent datatype for the columns from str to float. 
subset_df['Q7'] = subset_df.apply(lambda row: ', '.join([str(row['Q7_Part_1']), str(row['Q7_Part_2']), str(row['Q7_Part_3']), str(row['Q7_Part_4']),
                                           str(row['Q7_Part_5']),str(row['Q7_Part_6']),str(row['Q7_Part_7']),str(row['Q7_Part_8']),
                                           str(row['Q7_Part_9']),str(row['Q7_Part_10']),str(row['Q7_Part_11']),
                                           str(row['Q7_Part_12']),str(row['Q7_OTHER'])]), axis=1)
# Convert NaN values to empty string and then removing them from Q7 column
# Replace NaN values with an empty string
subset_df['Q7']= subset_df['Q7'].fillna('')

# Remove unwanted commas from the combined column
subset_df['Q7'] = subset_df['Q7'].str.replace(r'(^,)|(,$)|,(?=,)|nan, ', '').str.strip()
# the regular expressions (`(^,)|(,$)|,(?=,)|nan, `) are used in the `str.replace()` function to remove unwanted commas.
# The first part `(^,)` matches commas at the start of a string and replaces them with an empty string. 
# ($,) matches commas at the end of a string.
# ,(?=,) matches consecutive commas and replaces them with an empty string.
# Print the string for q7 to see if it needs more transformation
print(subset_df["Q7"])

1                            Python, R, nan
2                    SQL, C, C++, Java, nan
3                    Python, C++, Java, nan
4                               Python, nan
5                    Python, C, MATLAB, nan
                        ...                
25969    Python, SQL, Javascript, Bash, nan
25970                           Python, nan
25971                                   nan
25972                      Python, SQL, nan
25973                                   nan
Name: Q7, Length: 25973, dtype: object


<ipython-input-15-354536b4d2b0>:13: FutureWarning: The default value of regex will change from True to False in a future version.
  subset_df['Q7'] = subset_df['Q7'].str.replace(r'(^,)|(,$)|,(?=,)|nan, ', '').str.strip()


##### Repeat Step 3 Transformation of replacing nan values for the combined column

In [18]:
# Replace NaN values at the end along with the comma with an empty string
subset_df['Q7']=subset_df['Q7'].str.replace(', nan', '').str.strip()
subset_df = subset_df.replace('nan', '')
subset_df['Q7']

1                            Python, R
2                    SQL, C, C++, Java
3                    Python, C++, Java
4                               Python
5                    Python, C, MATLAB
                     ...              
25969    Python, SQL, Javascript, Bash
25970                           Python
25971                                 
25972                      Python, SQL
25973                                 
Name: Q7, Length: 25973, dtype: object

##### Repeat step 2: keep the combined response column and drop the rest

In [20]:
columns_to_keep = ['Q1', 'Q2', 'Q3','Q4','Q5','Q6',
                   'Q20','Q21','Q25','Q7']  
subset_df = subset_df[columns_to_keep]
# Print the subset DataFrame

print(subset_df.head())

      Q1   Q2         Q3                 Q4                       Q5  \
1  50-54  Man      India  Bachelor’s degree                    Other   
2  50-54  Man  Indonesia    Master’s degree  Program/Project Manager   
3  22-24  Man   Pakistan    Master’s degree        Software Engineer   
4  45-49  Man     Mexico    Doctoral degree       Research Scientist   
5  45-49  Man      India    Doctoral degree                    Other   

           Q6                        Q20                   Q21            Q25  \
1  5-10 years  Manufacturing/Fabrication      50-249 employees  25,000-29,999   
2   20+ years  Manufacturing/Fabrication  1000-9,999 employees  60,000-69,999   
3   1-3 years        Academics/Education  1000-9,999 employees         $0-999   
4   20+ years        Academics/Education  1000-9,999 employees  30,000-39,999   
5   < 1 years        Academics/Education      50-249 employees  30,000-39,999   

                  Q7  
1          Python, R  
2  SQL, C, C++, Java  
3  Python, 

#### Step 4: Headers with appropriate names

In [21]:
# Replace Headers with appropriate names
subset_df = subset_df.rename(columns={"Q1": "age(in_years)",
                       "Q2": "gender", "Q3": "country" , "Q4": "highest_lvl_educ", "Q5": "job_role" ,
                        "Q6": "programming_exp(in_years)" , 
                        "Q7" : "programming_lang", "Q20" : "curr_industry",
                         "Q21": "company_size" , "Q25": "salary"
                       })

# Print the DataFrame with updated column header
subset_df


,age(in_years),gender,country,highest_lvl_educ,job_role,programming_exp(in_years),curr_industry,company_size,salary,programming_lang
1,50-54,Man,India,Bachelor’s degree,Other,5-10 years,Manufacturing/Fabrication,50-249 employees,"25,000-29,999","Python, R"
2,50-54,Man,Indonesia,Master’s degree,Program/Project Manager,20+ years,Manufacturing/Fabrication,"1000-9,999 employees","60,000-69,999","SQL, C, C++, Java"
3,22-24,Man,Pakistan,Master’s degree,Software Engineer,1-3 years,Academics/Education,"1000-9,999 employees",$0-999,"Python, C++, Java"
4,45-49,Man,Mexico,Doctoral degree,Research Scientist,20+ years,Academics/Education,"1000-9,999 employees","30,000-39,999",Python
5,45-49,Man,India,Doctoral degree,Other,< 1 years,Academics/Education,50-249 employees,"30,000-39,999","Python, C, MATLAB"
...,...,...,...,...,...,...,...,...,...,...
25969,30-34,Man,Egypt,Bachelor’s degree,Data Analyst,1-3 years,Computers/Technology,"10,000 or more employees","15,000-19,999","Python, SQL, Javascript, Bash"
25970,22-24,Man,China,Master’s degree,Student,1-3 years,NaN,NaN,NaN,Python
25971,50-54,Man,Sweden,Doctoral degree,Research Scientist,I have never written code,Academics/Education,"1000-9,999 employees",$0-999,
25972,45-49,Man,United States of America,Master’s degree,Data Scientist,5-10 years,Online Service/Internet-based Services,"10,000 or more employees",NaN,"Python, SQL"


####  Step 5: Count the null values/missing values in the columns

In [22]:

# Calculate the number of NaN values for each column
nan_counts = subset_df.isna().sum()
# Print the number of NaN values for each column
print(nan_counts)
empty_strings =  subset_df.apply(lambda x: (x == '').sum())
print(f' \nThe missing values for the columns are: \n{empty_strings}')

age(in_years)                    0
gender                           0
country                          0
highest_lvl_educ                 0
job_role                         0
programming_exp(in_years)        0
curr_industry                 9648
company_size                  9722
salary                       10582
programming_lang                 0
dtype: int64
 
The missing values for the columns are: 
age(in_years)                   0
gender                          0
country                         0
highest_lvl_educ                0
job_role                        0
programming_exp(in_years)       0
curr_industry                   0
company_size                    0
salary                          0
programming_lang             1032
dtype: int64


####  Step 6: Identifying the unique values in the columns since most of the data is categircal data

In [23]:
# Find unique values in each column to identify
# if the match with the survey and find outliers or inconsistent 
# since the data is not numeric, graphing the data or using box plot wouldn't work
unique_values = subset_df.apply(lambda x: x.unique())
for column in unique_values.index:
    print(column, ":")
    count = 0
    for value in unique_values[column]:
        print("-", value)
        count += 1
        if count == 15:
            break
    print()# Print the unique values in each column

age(in_years) :
- 50-54
- 22-24
- 45-49
- 25-29
- 18-21
- 30-34
- 40-44
- 35-39
- 70+
- 55-59
- 60-69

gender :
- Man
- Woman
- Nonbinary
- Prefer not to say
- Prefer to self-describe

country :
- India
- Indonesia
- Pakistan
- Mexico
- Russia
- Turkey
- Australia
- Nigeria
- Greece
- Belgium
- Japan
- Egypt
- Singapore
- Brazil
- Poland

highest_lvl_educ :
- Bachelor’s degree
- Master’s degree
- Doctoral degree
- I prefer not to answer
- Some college/university study without earning a bachelor’s degree
- No formal education past high school
- Professional doctorate

job_role :
- Other
- Program/Project Manager
- Software Engineer
- Research Scientist
- Currently not employed
- Student
- Data Scientist
- Data Analyst
- Machine Learning Engineer
- Business Analyst
- Data Engineer
- Product Manager
- Statistician
- Developer Relations/Advocacy
- DBA/Database Engineer

programming_exp(in_years) :
- 5-10 years
- 20+ years
- 1-3 years
- < 1 years
- 3-5 years
- 10-20 years
- I have never wri

#### Step 7: Format the data into pretyy and tangible format  using string mething title()

In [24]:
# Format data first 
# title () : Capitalizes first character of each word and all other characters converted to lowercase
subset_df['age(in_years)'] = subset_df['age(in_years)'].str.title()
subset_df['gender'] = subset_df['gender'].str.title()
subset_df['country'] = subset_df['country'].str.title()
subset_df['highest_lvl_educ'] = subset_df['highest_lvl_educ'].str.title()
subset_df['salary'] = subset_df['salary'].str.title()
subset_df['company_size'] = subset_df['company_size'].str.title()
subset_df['programming_lang'] = subset_df['programming_lang'].str.title()
subset_df['programming_exp(in_years)'] = subset_df['programming_exp(in_years)'].str.title()
subset_df['job_role'] = subset_df['job_role'].str.title()
subset_df['curr_industry'] = subset_df['curr_industry'].str.title()
subset_df.head()

,age(in_years),gender,country,highest_lvl_educ,job_role,programming_exp(in_years),curr_industry,company_size,salary,programming_lang
1,50-54,Man,India,Bachelor’S Degree,Other,5-10 Years,Manufacturing/Fabrication,50-249 Employees,"25,000-29,999","Python, R"
2,50-54,Man,Indonesia,Master’S Degree,Program/Project Manager,20+ Years,Manufacturing/Fabrication,"1000-9,999 Employees","60,000-69,999","Sql, C, C++, Java"
3,22-24,Man,Pakistan,Master’S Degree,Software Engineer,1-3 Years,Academics/Education,"1000-9,999 Employees",$0-999,"Python, C++, Java"
4,45-49,Man,Mexico,Doctoral Degree,Research Scientist,20+ Years,Academics/Education,"1000-9,999 Employees","30,000-39,999",Python
5,45-49,Man,India,Doctoral Degree,Other,< 1 Years,Academics/Education,50-249 Employees,"30,000-39,999","Python, C, Matlab"


#### Step 8: Fix casing or inconsistent values

In [25]:
# Change inconsistent gender values to 'Other' since we are particualry emphazing on the gender
subset_df['gender'] = subset_df['gender'].replace(['Nonbinary', 'Prefer Not To Say','Prefer To Self-Describe'], 
                                                  'Other')
unique_values = subset_df['gender'].unique() #to see the values in gender 
print(unique_values)

['Man' 'Woman' 'Other']


#### Step 9: Conduct Fuzzy Matching

In [26]:
# List of job roles related to data
data_job_roles= ['Data Analyst', 'Data Scientist', 
                 'Data Engineer', 'Business Analyst', 'Machine Learning Engineer', 'Statistician','DBA/Database Engineer']
column_name = 'job_role'
# Fuzzy matching for each job role in the data-related column
# by calculating matching scores with each data-related job role

# Filter job roles in the column for fuzzy matching
column_job_roles = subset_df[subset_df[column_name].isin(data_job_roles)][column_name].unique()

# Fuzzy matching for each selected job role in the column
matching_scores = []
for role in column_job_roles:
    scores = [(data_role, fuzz.ratio(role, data_role)) for data_role in data_job_roles]
    matching_scores.append(scores)

# Print the matching scores for each selected job role in the column
for i, role in enumerate(column_job_roles):
    print(f"Selected Job Role: {role}")
    scores = matching_scores[i]
    scores.sort(key=lambda x: x[1], reverse=True)
    for match in scores[:3]:
        print(f"- Match: {match[0]}, Score: {match[1]}")
    print()

Selected Job Role: Data Scientist
- Match: Data Scientist, Score: 100
- Match: Data Analyst, Score: 62
- Match: Data Engineer, Score: 52

Selected Job Role: Data Analyst
- Match: Data Analyst, Score: 100
- Match: Data Scientist, Score: 62
- Match: Business Analyst, Score: 57

Selected Job Role: Machine Learning Engineer
- Match: Machine Learning Engineer, Score: 100
- Match: Data Engineer, Score: 58
- Match: DBA/Database Engineer, Score: 48

Selected Job Role: Business Analyst
- Match: Business Analyst, Score: 100
- Match: Data Analyst, Score: 57
- Match: Data Scientist, Score: 27

Selected Job Role: Data Engineer
- Match: Data Engineer, Score: 100
- Match: DBA/Database Engineer, Score: 76
- Match: Machine Learning Engineer, Score: 58

Selected Job Role: Statistician
- Match: Statistician, Score: 100
- Match: Data Scientist, Score: 46
- Match: Data Analyst, Score: 33



#### Ethical Implications of Data Wrangling

In [27]:
# The data wrangling steps performed above involved several processes, 
# such as shortening the dataset, updating the gender column, and handling missing values.
# While these steps can improve data quality and analysis, they also raise ethical considerations. 
# When shortening the dataset, it is important to ensure that the process is conducted in a fair and unbiased manner,
# avoiding any discrimination or exclusion of certain individuals or groups.
# Additionally, when updating the gender column,it is important to minimize the potential bias 
# that would arise from not considering all gender entires.
# From KAGGLE, The following methodolgy was included when acquiring the dataset.
#To ensure response quality, we excluded respondents that were flagged by our survey system as
#“Spam” or "Duplicate". We also dropped responses from respondents that spent less than 2
# minutes completing the survey, as well as responses from respondents that selected fewer than
# 15 answer choices in total.
# To protect the respondents’ privacy, free-form text responses were not included in the public
#survey dataset, and the order of the rows was shuffled (responses are not displayed in
#chronological order). Likewise, if a country or territory received less than 50 respondents, we
#grouped them into a group named “Other” for the sake of anonymity.
